In [1]:
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import networkx as nx
from functions.utilities import preprocess
from functions.analysis import generate_graph_figure


In [2]:
filename='processed_data\itigi_comments_copnsolidated.parqet'
litigi=pd.read_parquet(filename).drop_duplicates(subset=['author','body','id', 'created_utc'])
litigi['created_dt']=(pd.to_datetime(litigi['created_utc'],unit='s'))
litigi['body_preprocessed']=litigi['body'].apply(lambda x: preprocess(x))
id_to_author = litigi.set_index('id')['author'].to_dict()
litigi['parent_author'] = litigi['parent_id'].str.split('_').str[1].map(id_to_author)
litigi.head()

,author,author_flair_css_class,author_flair_text,body,can_gild,collapsed,collapsed_reason,controversiality,created_utc,distinguished,...,top_awarded_type,comment_type,collapsed_reason_code,retrieved_utc,author_is_blocked,unrepliable_reason,editable,created_dt,body_preprocessed,parent_author
0,Trickyhere,None,None,Qualcosa da spaccare?,True,False,None,0,1502873647,None,...,None,None,None,NaN,None,NaN,None,2017-08-16 08:54:07,Qualcosa da spaccare?,NaN
1,panicClark,None,None,"Avevo quest'ideona dello sticky giornaliero ""A...",True,False,None,0,1502873820,None,...,None,None,None,NaN,None,NaN,None,2017-08-16 08:57:00,"Avevo quest'ideona dello sticky giornaliero ""A...",NaN
2,MeccAnon,None,None,"Risposta USENET: ""non l'avete capita"" (cit.)\n...",True,False,None,0,1502874353,None,...,None,None,None,NaN,None,NaN,None,2017-08-16 09:05:53,"Risposta USENET: ""non l'avete capita"" (cit.) ...",panicClark
3,panicClark,None,None,[GENIO.](https://www.reddit.com/r/litigi/comme...,True,False,None,0,1502876399,moderator,...,None,None,None,NaN,None,NaN,None,2017-08-16 09:39:59,GENIO.,MeccAnon
4,MeccAnon,None,None,&gt; nella sidebar trovate il suo manifesto or...,True,False,None,0,1502877513,None,...,None,None,None,NaN,None,NaN,None,2017-08-16 09:58:33,&gt; nella sidebar trovate il suo manifesto or...,NaN


In [ ]:
df_pivot = pd.pivot_table(
    litigi,
    index='author',
    columns="sentiment",
    aggfunc='size'
)

In [ ]:
df_pivot=df_pivot.fillna(0)
df_pivot=df_pivot.sort_values('positive',ascending=False)
df_pivot['positive_ratio']=df_pivot['positive']/(df_pivot['negative']+df_pivot['neutral'])
df_pivot['p/n']=df_pivot['positive']/(df_pivot['negative'])

In [ ]:
df_pivot

In [ ]:
litigi[(litigi['num_words']==1) & (litigi['body_preprocessed']!='[deleted]')][['body_preprocessed']]

In [ ]:
df_pivot.to_csv('/content/drive/MyDrive/AI/data/output/litigi_sentiment.csv')

In [ ]:
litigi[(litigi['body_preprocessed']!='[deleted]') &(litigi['body_preprocessed']!='[removed]') ]['num_words'].hist(bins=range(75),grid=False)

In [ ]:
sns.kdeplot(litigi[ (litigi['num_words']<100) &(litigi['num_words']>1) ]['num_words'])
plt.show()

In [ ]:
author_list=['tommyrugby','sda_express','SpiegoLeDiscussioni','PHEELZ']

In [ ]:
df_filtered = litigi[litigi['author'].isin(author_list)]

In [ ]:
for author in author_list:
    plt.hist(df_filtered[df_filtered['author'] == author]['num_words'], alpha=0.5, label=author,bins=range(75))
plt.legend(loc='upper right')
plt.xlabel('Number of words')
plt.ylabel('Frequency')
plt.show()

In [ ]:

histograms = []
for author in author_list:
    author_hist, _ = np.histogram(df_filtered[df_filtered['author'] == author]['num_words'],bins=range(50))
    histograms.append(author_hist)

# Plot histograms side by side
plt.figure(figsize=(19, 10))
width = 0.2
x = np.arange(len(histograms[0]))

for i, author_hist in enumerate(histograms):
    plt.bar(x + i*width, author_hist, width=width, alpha=0.9, label=author_list[i])

plt.xlabel('Number of words')
plt.ylabel('Frequency')
plt.legend(loc='upper right')
plt.show()


In [ ]:
date=pd.to_datetime(litigi['created_utc'], unit='s').dt.date
date

In [ ]:
plt.hist(date,bins=50,alpha=0.5)

plt.show()

In [ ]:
time=pd.to_datetime(litigi['created_utc'], unit='s').dt.hour
time

In [ ]:
plt.hist(time,alpha=0.9,rwidth=0.7,bins=24)

plt.show()

In [ ]:
histograms = []
for author in author_list:
    author_hist, _ = np.histogram(pd.to_datetime(df_filtered[df_filtered['author'] == author]['created_utc'], unit='s').dt.hour,bins=24,density=True)
    histograms.append(author_hist)

# Plot histograms side by side
plt.figure(figsize=(19, 10))
width = 0.2
x = np.arange(len(histograms[0]))

for i, author_hist in enumerate(histograms):
    plt.bar(x + i*width, author_hist, width=width, alpha=0.9, label=author_list[i])

plt.xlabel('hour')
plt.ylabel('Frequency')
plt.legend(loc='upper right')
plt.show()

In [ ]:
df_filtered = litigi[litigi['author'].isin(author_list)]

In [ ]:
plt.figure(figsize=(19, 10))

for author in author_list:
    sns.kdeplot(pd.to_datetime(df_filtered[df_filtered['author'] == author]['created_utc'], unit='s').dt.hour,label=author)
    #histograms.append(author_hist)

plt.xlabel('hour')
plt.ylabel('Frequency')
plt.legend(loc='upper right')
plt.show()

In [ ]:
plt.figure(figsize=(19, 10))

sns.kdeplot(data=df_filtered,x=pd.to_datetime(df_filtered['created_utc'], unit='s').dt.hour,label=author,hue='author', common_norm=False)
plt.show()